# GradientBoostingClassifier

Pipeline hoàn chỉnh trong một notebook:
1. Input dữ liệu `train/val/test`
2. Preprocess
3. Normalize
4. Chọn `X` và `y`
5. Train trên `train.csv`
6. Validate trên `val.csv`
7. Nếu chưa tốt thì thử lại với bộ tham số khác
8. Refit model tốt nhất trên `train + val`
9. Lưu model
10. Đánh giá mô hình
11. Predict trên `test.csv`

In [1]:
from pathlib import Path
import json

import joblib
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import GradientBoostingClassifier

MODEL_NAME = "GradientBoostingClassifier"
TARGET_COLUMN = "target"
DATA_DIR = Path("../data")
ARTIFACTS_DIR = Path("../notebook_outputs")
PARAM_GRID = [{'n_estimators': 100, 'learning_rate': 0.05, 'max_depth': 2}, {'n_estimators': 150, 'learning_rate': 0.1, 'max_depth': 2}, {'n_estimators': 200, 'learning_rate': 0.05, 'max_depth': 3}]


def parse_numeric_value(value):
    if pd.isna(value):
        return value
    if isinstance(value, str):
        cleaned = value.strip()
        if cleaned.count(".") > 1:
            sign = -1 if cleaned.startswith("-") else 1
            digits_only = cleaned[1:] if sign == -1 else cleaned
            digits_only = digits_only.replace(".", "")
            return sign * float(f"0.{digits_only}")
        return float(cleaned)
    return float(value)


def load_dataset(file_path):
    df = pd.read_csv(file_path)
    object_columns = df.select_dtypes(include="object").columns
    for col in object_columns:
        df[col] = df[col].map(parse_numeric_value)
    return df


def split_xy(df):
    x = df.drop(columns=[TARGET_COLUMN]).copy()
    y = df[TARGET_COLUMN].copy()
    return x, y


def build_preprocessor(feature_names):
    numeric_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", MinMaxScaler()),
        ]
    )
    return ColumnTransformer(
        transformers=[("num", numeric_pipeline, feature_names)],
        remainder="drop",
    )


def build_model(params):
    estimator = clone(GradientBoostingClassifier(random_state=42)).set_params(**params)
    return Pipeline(
        steps=[
            ("preprocess", build_preprocessor(feature_names)),
            ("model", estimator),
        ]
    )


def get_scores(model, x):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(x)[:, 1]
    if hasattr(model, "decision_function"):
        return model.decision_function(x)
    return None


def evaluate_model(model, x, y):
    y_pred = model.predict(x)
    y_score = get_scores(model, x)
    metrics = {
        "accuracy": float(accuracy_score(y, y_pred)),
        "precision": float(precision_score(y, y_pred, zero_division=0)),
        "recall": float(recall_score(y, y_pred, zero_division=0)),
        "f1": float(f1_score(y, y_pred, zero_division=0)),
        "confusion_matrix": confusion_matrix(y, y_pred).tolist(),
    }
    if y_score is not None and len(np.unique(y)) > 1:
        metrics["roc_auc"] = float(roc_auc_score(y, y_score))
    return metrics


train_df = load_dataset(DATA_DIR / "train.csv")
val_df = load_dataset(DATA_DIR / "val.csv")
test_df = load_dataset(DATA_DIR / "test.csv")

x_train, y_train = split_xy(train_df)
x_val, y_val = split_xy(val_df)
x_test, y_test = split_xy(test_df)
feature_names = x_train.columns.tolist()

best_result = None

for params in PARAM_GRID:
    pipeline = build_model(params)
    pipeline.fit(x_train, y_train)
    train_metrics = evaluate_model(pipeline, x_train, y_train)
    val_metrics = evaluate_model(pipeline, x_val, y_val)
    current_result = {
        "params": params,
        "train_metrics": train_metrics,
        "val_metrics": val_metrics,
        "model": pipeline,
    }
    if best_result is None:
        best_result = current_result
    else:
        current_key = (current_result["val_metrics"]["f1"], current_result["val_metrics"]["accuracy"])
        best_key = (best_result["val_metrics"]["f1"], best_result["val_metrics"]["accuracy"])
        if current_key > best_key:
            best_result = current_result

print("Best params on validation:", best_result["params"])
print("Train metrics:", json.dumps(best_result["train_metrics"], indent=2))
print("Validation metrics:", json.dumps(best_result["val_metrics"], indent=2))

x_train_val = pd.concat([x_train, x_val], ignore_index=True)
y_train_val = pd.concat([y_train, y_val], ignore_index=True)

final_model = build_model(best_result["params"])
final_model.fit(x_train_val, y_train_val)

train_val_metrics = evaluate_model(final_model, x_train_val, y_train_val)
test_metrics = evaluate_model(final_model, x_test, y_test)
test_predictions = final_model.predict(x_test)

models_dir = ARTIFACTS_DIR / "models"
reports_dir = ARTIFACTS_DIR / "reports"
predictions_dir = ARTIFACTS_DIR / "predictions"
models_dir.mkdir(parents=True, exist_ok=True)
reports_dir.mkdir(parents=True, exist_ok=True)
predictions_dir.mkdir(parents=True, exist_ok=True)

model_path = models_dir / f"{MODEL_NAME}.joblib"
report_path = reports_dir / f"{MODEL_NAME}_metrics.json"
prediction_path = predictions_dir / f"{MODEL_NAME}_test_predictions.csv"

joblib.dump(final_model, model_path)

prediction_df = x_test.copy()
prediction_df["actual_target"] = y_test.values
prediction_df["predicted_target"] = test_predictions
prediction_df.to_csv(prediction_path, index=False)

report = {
    "model_name": MODEL_NAME,
    "best_validation_params": best_result["params"],
    "train_metrics": best_result["train_metrics"],
    "validation_metrics": best_result["val_metrics"],
    "train_val_metrics_after_refit": train_val_metrics,
    "test_metrics": test_metrics,
    "saved_model_path": str(model_path),
    "prediction_path": str(prediction_path),
}

report_path.write_text(json.dumps(report, indent=2), encoding="utf-8")

print("\nTrain+Val metrics after refit:")
print(json.dumps(train_val_metrics, indent=2))
print("\nTest metrics:")
print(json.dumps(test_metrics, indent=2))
print("\nSaved model:", model_path)
print("Saved report:", report_path)
print("Saved predictions:", prediction_path)

prediction_df.head()


Best params on validation: {'n_estimators': 100, 'learning_rate': 0.05, 'max_depth': 2}
Train metrics: {
  "accuracy": 0.9132231404958677,
  "precision": 0.9245283018867925,
  "recall": 0.8828828828828829,
  "f1": 0.9032258064516129,
  "confusion_matrix": [
    [
      123,
      8
    ],
    [
      13,
      98
    ]
  ],
  "roc_auc": 0.9656832404924008
}
Validation metrics: {
  "accuracy": 0.9,
  "precision": 0.8235294117647058,
  "recall": 1.0,
  "f1": 0.9032258064516129,
  "confusion_matrix": [
    [
      13,
      3
    ],
    [
      0,
      14
    ]
  ],
  "roc_auc": 0.9776785714285714
}

Train+Val metrics after refit:
{
  "accuracy": 0.9044117647058824,
  "precision": 0.9159663865546218,
  "recall": 0.872,
  "f1": 0.8934426229508197,
  "confusion_matrix": [
    [
      137,
      10
    ],
    [
      16,
      109
    ]
  ],
  "roc_auc": 0.9635374149659864
}

Test metrics:
{
  "accuracy": 0.8387096774193549,
  "precision": 0.8,
  "recall": 0.8571428571428571,
  "f1": 0.8275

,age,trestbps,chol,thalach,oldpeak,sex,cp,fbs,restecg,exang,slope,ca,thal,actual_target,predicted_target
0,0.384303,-0.168240,-0.641646,-0.837597,0.107158,1.0,1.000000,0.0,1.0,1.0,0.5,1.0,1.0,1,1
1,-0.228879,-0.736870,-0.128635,0.106174,-0.891627,1.0,0.000000,0.0,1.0,0.0,0.0,0.0,0.0,0,0
2,0.829818,-0.545134,-0.357219,-0.175039,0.714629,1.0,0.666667,0.0,0.0,0.0,0.5,1.0,1.0,0,1
3,-0.395349,-0.545134,0.116827,-0.425278,-0.445445,0.0,0.666667,0.0,1.0,0.0,0.0,0.0,0.0,0,0
4,-0.139776,-0.623144,-0.186562,0.194515,-0.177735,1.0,0.666667,1.0,0.0,0.0,1.0,0.0,1.0,0,0
